# XGBoost with Class-Based Balancing

This notebook trains an XGBoost model for heart disease prediction using class-based weighting through `scale_pos_weight` to handle class imbalance.

In [1]:
from pathlib import Path
import pickle

import optuna
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,  log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import warnings                       
warnings.filterwarnings('ignore') 

In [2]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for path in [current, *current.parents]:
        # Use the data folder to identify the main project directory.
        if (path / 'data').exists():
            return path

    raise FileNotFoundError('Project root not found.')


# Define all important paths used in this notebook.
BASE_DIR = find_project_root(Path.cwd())
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'heart_disease_processed.csv'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / 'xgboost_model.pkl'
COLUMNS_PATH = MODEL_DIR / 'xgboost_columns.pkl'

BASE_DIR, DATA_PATH, MODEL_PATH

(WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/processed/heart_disease_processed.csv'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/models/xgboost_model.pkl'))

In [3]:
# Load the processed dataset and keep a working copy for training.
data = pd.read_csv(DATA_PATH)
df = data.copy()
df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40.0,1,0,0.0,0,0,...,1,0,5.0,18.0,15.0,1,0,9.0,4.0,3.0
1,0,0,0,0,25.0,1,0,0.0,1,0,...,0,1,3.0,0.0,0.0,0,0,7.0,6.0,1.0
2,0,1,1,1,28.0,0,0,0.0,0,1,...,1,1,5.0,30.0,30.0,1,0,9.0,4.0,8.0
3,0,1,0,1,27.0,0,0,0.0,1,1,...,1,0,2.0,0.0,0.0,0,0,11.0,3.0,6.0
4,0,1,1,1,24.0,0,0,0.0,1,1,...,1,0,2.0,3.0,0.0,0,0,11.0,5.0,4.0


In [4]:
# Separate the target column from the feature columns.
target_column = 'HeartDiseaseorAttack'

X = df.drop(columns=[target_column])
y = df[target_column]

print(f'Feature shape: {X.shape}')
print(f'Target shape: {y.shape}')

Feature shape: (229781, 21)
Target shape: (229781,)


In [5]:
# Check the class balance before training the model.
class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize=True).sort_index() * 100

print('Class counts:')
print(class_counts)
print('\nClass percentages:')
print(class_percentages.round(2))

Class counts:
HeartDiseaseorAttack
0    206064
1     23717
Name: count, dtype: int64

Class percentages:
HeartDiseaseorAttack
0    89.68
1    10.32
Name: proportion, dtype: float64


In [6]:
# Split the data before training so the test data remains unseen.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (183824, 21)
X_test shape: (45957, 21)


In [7]:
# Compute the class ratio so XGBoost can give more weight to positive-class errors.
negative_class_count = (y_train == 0).sum()
positive_class_count = (y_train == 1).sum()
scale_pos_weight = negative_class_count / positive_class_count

print(f'scale_pos_weight: {scale_pos_weight:.4f}')

scale_pos_weight: 8.6882


In [8]:
# Use scale_pos_weight as the class-based imbalance handling method for XGBoost.
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [9]:
# Generate predictions and prediction probabilities for evaluation.
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

In [10]:
# Evaluate the model using multiple metrics.
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.7370
ROC-AUC Score: 0.8350

Confusion Matrix:
[[30169 11045]
 [ 1041  3702]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.73      0.83     41214
           1       0.25      0.78      0.38      4743

    accuracy                           0.74     45957
   macro avg       0.61      0.76      0.61     45957
weighted avg       0.89      0.74      0.79     45957



In [11]:
study = optuna.create_study(direction="minimize")  

[I 2026-04-09 03:08:54,751] A new study created in memory with name: no-name-8e024562-0565-470e-a8de-badfe969f375


In [12]:
for _ in range(50):
    trial = study.ask()

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    loss = log_loss(y_test, y_prob)

    study.tell(trial, loss)

In [13]:
best = study.best_params

# Final model
tuned_model = XGBClassifier(**best, random_state=42, use_label_encoder=False, eval_metric="logloss")
tuned_model.fit(X_train, y_train)

y_pred = tuned_model.predict(X_test)

print("Best Params:", best)
print("Accuracy:", accuracy_score(y_test, y_pred))

Best Params: {'n_estimators': 113, 'max_depth': 4, 'learning_rate': 0.0880691135709043, 'subsample': 0.797620443022196, 'colsample_bytree': 0.6287840789277795, 'scale_pos_weight': 1.0712643956764618}
Accuracy: 0.8987749417934155


In [14]:
# Evaluate the model using multiple metrics.
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.8988
ROC-AUC Score: 0.8350

Confusion Matrix:
[[40748   466]
 [ 4186   557]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95     41214
           1       0.54      0.12      0.19      4743

    accuracy                           0.90     45957
   macro avg       0.73      0.55      0.57     45957
weighted avg       0.87      0.90      0.87     45957



In [15]:
# Save the trained model and feature columns for later use in prediction.
with open(MODEL_PATH, 'wb') as model_file:
    pickle.dump(xgb_model, model_file)

with open(COLUMNS_PATH, 'wb') as columns_file:
    pickle.dump(X.columns.tolist(), columns_file)

print(f'Model saved to: {MODEL_PATH}')
print(f'Columns saved to: {COLUMNS_PATH}')

Model saved to: D:\PROJECT OF DATA SCIENCE\Heart Disease Health Indicators\models\xgboost_model.pkl
Columns saved to: D:\PROJECT OF DATA SCIENCE\Heart Disease Health Indicators\models\xgboost_columns.pkl


In [16]:
# Review feature importance to understand which variables influenced the model most.
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance_df

,feature,importance
0,HighBP,0.255155
13,GenHlth,0.107934
17,Sex,0.102993
18,Age,0.100106
16,DiffWalk,0.083977
5,Stroke,0.080610
1,HighChol,0.073999
4,Smoker,0.028736
6,Diabetes,0.025079
2,CholCheck,0.015982
